In [1]:
import pandas as pd
import numpy as np

In [2]:
p1 = pd.read_csv("validation_part_1_labelled.csv", dtype={"id": str}, encoding="latin-1")
p2 = pd.read_csv("validation_part_2_labelled.csv", dtype={"id": str}, encoding="latin-1")
p3 = pd.read_csv("validation_part_3_labelled.csv", dtype={"id": str}, encoding="latin-1")
p4 = pd.read_csv("validation_part_4_labelled.csv", dtype={"id": str}, encoding="latin-1")
val = pd.concat([p1, p2, p3, p4], ignore_index=True)
print(f"Total rows: {len(val)}")

Total rows: 1199


In [3]:
# parse semicolon-separated string into a normalized set
def parse_locations(text):
    """Convert 'Edinburgh Castle; Royal Mile' into {'edinburgh castle', 'royal mile'}"""
    if pd.isna(text) or str(text).strip() == "":
        return set()
    return set(item.strip().lower() for item in str(text).split(";") if item.strip())

In [4]:
def compute_token_f1(df, llm_col, manual_col):
    total_tp = total_fp = total_fn = 0
    for _, row in df.iterrows():
        llm_set = parse_locations(row[llm_col])
        manual_set = parse_locations(row[manual_col])
        total_tp += len(llm_set & manual_set)
        total_fp += len(llm_set - manual_set)
        total_fn += len(manual_set - llm_set)
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"TP": total_tp, "FP": total_fp, "FN": total_fn,
            "Precision": round(precision, 4), "Recall": round(recall, 4), "F1": round(f1, 4)}

In [5]:
def compute_binary_f1(df, llm_col, manual_col):
    llm_bin = df[llm_col].apply(lambda x: 1 if pd.notna(x) and str(x).strip() != "" else 0)
    man_bin = df[manual_col].apply(lambda x: 1 if pd.notna(x) and str(x).strip() != "" else 0)
    tp = ((llm_bin == 1) & (man_bin == 1)).sum()
    fp = ((llm_bin == 1) & (man_bin == 0)).sum()
    fn = ((llm_bin == 0) & (man_bin == 1)).sum()
    tn = ((llm_bin == 0) & (man_bin == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
            "Precision": round(precision, 4), "Recall": round(recall, 4), "F1": round(f1, 4)}

In [6]:
# comparisons
categories = [
    ("specific", "llm_specific", "manual_specific"),
    ("general",  "llm_general",  "manual_general"),
    ("parent",   "llm_parent",   "manual_parent"),
]
 
print("\n" + "=" * 60)
print("BINARY F1 (presence vs absence)")
print("=" * 60)
for name, llm_col, manual_col in categories:
    r = compute_binary_f1(val, llm_col, manual_col)
    print(f"\n  {name.upper()}:")
    print(f"    TP={r['TP']}  FP={r['FP']}  FN={r['FN']}  TN={r['TN']}")
    print(f"    Precision={r['Precision']}  Recall={r['Recall']}  F1={r['F1']}")
 
print("\n" + "=" * 60)
print("TOKEN-LEVEL F1 (individual location matching)")
print("=" * 60)
for name, llm_col, manual_col in categories:
    r = compute_token_f1(val, llm_col, manual_col)
    print(f"\n  {name.upper()}:")
    print(f"    TP={r['TP']}  FP={r['FP']}  FN={r['FN']}")
    print(f"    Precision={r['Precision']}  Recall={r['Recall']}  F1={r['F1']}")


BINARY F1 (presence vs absence)

  SPECIFIC:
    TP=897  FP=55  FN=21  TN=226
    Precision=0.9422  Recall=0.9771  F1=0.9594

  GENERAL:
    TP=986  FP=40  FN=40  TN=133
    Precision=0.961  Recall=0.961  F1=0.961

  PARENT:
    TP=322  FP=155  FN=113  TN=609
    Precision=0.6751  Recall=0.7402  F1=0.7061

TOKEN-LEVEL F1 (individual location matching)

  SPECIFIC:
    TP=2780  FP=987  FN=522
    Precision=0.738  Recall=0.8419  F1=0.7865

  GENERAL:
    TP=3074  FP=1192  FN=1180
    Precision=0.7206  Recall=0.7226  F1=0.7216

  PARENT:
    TP=310  FP=228  FN=151
    Precision=0.5762  Recall=0.6725  F1=0.6206


## **The whole logic about the *F1 SCORE* calculation**

Each listing description was processed twice: once by the LLM and once by manual labelling. Both produce three categories of output:

Specific — named, mappable places (e.g. "Edinburgh Castle", "Royal Mile")

General — generic place types (e.g. "shops", "pubs", "city centre")

Parent — the neighborhood where the property is located (e.g. "Leith")

What we do currently is to measure how well the LLM's output matches the manual labels. And in this version, I calculate F1 at two levels and they answer different questions separately.

Binary F1 asks: "Did the LLM detect any location in this category, yes or no?" For each row, simply check whether the column is empty or not. If both LLM and manual have content, that's a **TP**. If LLM has content but manual is empty, that's an **FP**. And so on. This tells us whether the LLM can recognize the presence of locations.

Token-level F1 asks: "Did the LLM identify the exact same locations as the manual label?" Here split each semicolon-separated string into a set of individual items and compare them. For example, if LLM outputs "Edinburgh Castle; Royal Mile" and manual says "Edinburgh Castle; Grassmarket", then Edinburgh Castle is a **TP** (both found it), Royal Mile is an **FP** (LLM found it but manual didn't), and Grassmarket is an **FN** (manual found it but LLM didn't).

## **Results analysis**

**Binary F1: Specific (F1=0.959)**
The LLM correctly identifies whether a description mentions named places in 96% of cases. Recall is 0.977, meaning the LLM very rarely misses descriptions containing specific locations. Only 55 FPs suggest occasional over-detection.

**Binary F1: General (F1=0.961)**
The strongest binary score. FP and FN are perfectly balanced at 40 each, indicating the LLM is highly reliable at detecting generic place references like "shops", "restaurants", and "city centre".

**Binary F1: Parent (F1=0.706)**
The weakest category. FP=155 remains high, meaning the LLM sometimes identifies a parent neighbourhood when the manual label disagrees.

**Token-level F1: Specific (F1=0.786)**
The strongest token-level score. Recall=0.842 shows the LLM captures most named places, and Precision=0.738 indicates reasonable accuracy. Remaining mismatches are likely due to naming differences (e.g. "Botanic Gardens" vs "Royal Botanic Garden").

**Token-level F1: General (F1=0.722)**
Both FP and FN are around 1,100–1,200, reflecting some disagreement on what counts as a "general" location. The LLM may classify items like "Old Town" as general, while the manual label treats them as specific or parent.

**Token-level F1: Parent (F1=0.621)**
310 true positives against 228 false positives and 151 false negatives. It' s the weakest token-level category, confirming that parent neighbourhood identification remains the LLM's biggest challenge.

In [7]:
# Combined F1 across all three categories
print("\n" + "=" * 60)
print("COMBINED F1 (specific + general + parent)")
print("=" * 60)

# Binary combined
b_tp = b_fp = b_fn = b_tn = 0
for name, llm_col, manual_col in categories:
    r = compute_binary_f1(val, llm_col, manual_col)
    b_tp += r["TP"]; b_fp += r["FP"]; b_fn += r["FN"]; b_tn += r["TN"]
 
b_precision = b_tp / (b_tp + b_fp) if (b_tp + b_fp) > 0 else 0
b_recall    = b_tp / (b_tp + b_fn) if (b_tp + b_fn) > 0 else 0
b_f1        = 2 * b_precision * b_recall / (b_precision + b_recall) if (b_precision + b_recall) > 0 else 0
 
print(f"\n  Binary:")
print(f"    TP={b_tp}  FP={b_fp}  FN={b_fn}  TN={b_tn}")
print(f"    Precision={b_precision:.4f}  Recall={b_recall:.4f}  F1={b_f1:.4f}")


COMBINED F1 (specific + general + parent)

  Binary:
    TP=2205  FP=250  FN=174  TN=968
    Precision=0.8982  Recall=0.9269  F1=0.9123


In [8]:
# Token-level combined
t_tp = t_fp = t_fn = 0
for name, llm_col, manual_col in categories:
    r = compute_token_f1(val, llm_col, manual_col)
    t_tp += r["TP"]; t_fp += r["FP"]; t_fn += r["FN"]
 
t_precision = t_tp / (t_tp + t_fp) if (t_tp + t_fp) > 0 else 0
t_recall    = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
t_f1        = 2 * t_precision * t_recall / (t_precision + t_recall) if (t_precision + t_recall) > 0 else 0
 
print(f"\n  Token-level:")
print(f"    TP={t_tp}  FP={t_fp}  FN={t_fn}")
print(f"    Precision={t_precision:.4f}  Recall={t_recall:.4f}  F1={t_f1:.4f}")


  Token-level:
    TP=6164  FP=2407  FN=1853
    Precision=0.7192  Recall=0.7689  F1=0.7432


**Combined F1:**
Binary F1 = 0.912, Token-level F1 = 0.743 across all three categories, indicating strong overall LLM performance with 1,199 validated listings.

In [1]:
!jupyter nbconvert --to html F1_Score2.0.ipynb

[NbConvertApp] Converting notebook F1_Score2.0.ipynb to html
[NbConvertApp] Writing 320785 bytes to F1_Score2.0.html
